In [11]:
!pip install torchmetrics
!pip install tqdm

In [12]:
import torch
import torch.nn as nn
import numpy as np

def patchify(images, n_patches):
    n, c, h, w = images.shape
    assert h == w, "Patchify method is implemented for square images only"
    patches = torch.zeros(n, n_patches ** 2, h * w * c // n_patches ** 2, device=images.device)
    patch_size = h // n_patches
    for idx, image in enumerate(images):
        for i in range(n_patches):
            for j in range(n_patches):
                patch = image[:, i * patch_size: (i + 1) * patch_size, j * patch_size: (j + 1) * patch_size]
                patches[idx, i * n_patches + j] = patch.flatten()
    return patches

def get_positional_embeddings(sequence_length, d):
    result = torch.ones(sequence_length, d)
    for i in range(sequence_length):
        for j in range(d):
            result[i][j] = np.sin(i / (10000 ** (j / d))) if j % 2 == 0 else np.cos(i / (10000 ** ((j - 1) / d)))
    return result

class MyMSA(nn.Module):
    def __init__(self, d, n_heads=2):
        super(MyMSA, self).__init__()
        self.d = d
        self.n_heads = n_heads
        assert d % n_heads == 0, f"Can't divide dimension {d} into {n_heads} heads"

        self.d_head = d // n_heads
        self.q_mappings = nn.ModuleList([nn.Linear(self.d_head, self.d_head) for _ in range(n_heads)])
        self.k_mappings = nn.ModuleList([nn.Linear(self.d_head, self.d_head) for _ in range(n_heads)])
        self.v_mappings = nn.ModuleList([nn.Linear(self.d_head, self.d_head) for _ in range(n_heads)])
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, sequences):
        # Sequences shape: (N, seq_len, d)
        result = []
        for sequence in sequences:
            local_result = []
            for h in range(self.n_heads):
                q_mapping = self.q_mappings[h]
                k_mapping = self.k_mappings[h]
                v_mapping = self.v_mappings[h]

                seq = sequence[:, h * self.d_head : (h + 1) * self.d_head]
                q = q_mapping(seq)
                k = k_mapping(seq)
                v = v_mapping(seq)

                attention = self.softmax(q @ k.T / (self.d_head ** 0.5))
                local_result.append(attention @ v)
            result.append(torch.hstack(local_result))
        return torch.stack(result)

class MyViTBlock(nn.Module):
    def __init__(self, hidden_d, n_heads, mlp_ratio=4):
        super(MyViTBlock, self).__init__()
        self.hidden_d = hidden_d
        self.n_heads = n_heads

        self.norm1 = nn.LayerNorm(hidden_d)
        self.mhsa = MyMSA(hidden_d, n_heads)
        self.norm2 = nn.LayerNorm(hidden_d)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_d, mlp_ratio * hidden_d),
            nn.GELU(),
            nn.Linear(mlp_ratio * hidden_d, hidden_d)
        )

    def forward(self, x):
        out = x + self.mhsa(self.norm1(x))
        out = out + self.mlp(self.norm2(out))
        return out

class MyViT(nn.Module):
    def __init__(self, chw=(1, 28, 28), n_patches=7, n_blocks=2, hidden_d=8, n_heads=2, out_d=10):
        super(MyViT, self).__init__()
        self.chw = chw
        self.n_patches = n_patches
        self.n_blocks = n_blocks
        self.hidden_d = hidden_d
        self.n_heads = n_heads
        self.out_d = out_d

        assert chw[1] % n_patches == 0, "Input shape not entirely divisible by number of patches"
        assert chw[2] % n_patches == 0, "Input shape not entirely divisible by number of patches"
        self.patch_size = (chw[1] / n_patches, chw[2] / n_patches)

        # 1. Linear mapper
        self.input_d = int(chw[0] * self.patch_size[0] * self.patch_size[1])
        self.linear_mapper = nn.Linear(self.input_d, self.hidden_d)

        # 2. Learnable classification token
        self.class_token = nn.Parameter(torch.rand(1, self.hidden_d))

        # 3. Positional embedding
        self.register_buffer('positional_embeddings', get_positional_embeddings(n_patches ** 2 + 1, hidden_d), persistent=False)

        # 4. Transformer encoder blocks
        self.blocks = nn.ModuleList([MyViTBlock(hidden_d, n_heads) for _ in range(n_blocks)])

        # 5. Classification MLP head
        self.mlp = nn.Sequential(
            nn.Linear(self.hidden_d, self.out_d)
        )

    def forward(self, images):
        n, c, h, w = images.shape
        patches = patchify(images, self.n_patches)
        tokens = self.linear_mapper(patches)

        # Ajout du token de classification [CLS]
        tokens = torch.cat((self.class_token.expand(n, 1, -1), tokens), dim=1)

        # Ajout des positional embeddings
        out = tokens + self.positional_embeddings.to(images.device)

        # Passage à travers les blocs encodeurs Transformer
        for block in self.blocks:
            out = block(out)

        # Extraction du token de classification uniquement (index 0)
        out = out[:, 0]

        # Classification finale
        return self.mlp(out)

In [13]:
import time
from torch.optim import Adam
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader
from torchvision.transforms import ToTensor
from torchvision.datasets.mnist import MNIST
from torchmetrics.classification import Accuracy, F1Score
# Remplace ton importation actuelle de tqdm par celle-ci :
from tqdm.notebook import tqdm, trange

# Reproductibilité
torch.manual_seed(0)

def main():
    # 1. Chargement des données MNIST
    transform = ToTensor()
    train_set = MNIST(root='./data', train=True, download=True, transform=transform)
    test_set = MNIST(root='./data', train=False, download=True, transform=transform)

    train_loader = DataLoader(train_set, shuffle=True, batch_size=128)
    test_loader = DataLoader(test_set, shuffle=False, batch_size=128)

    # 2. Configuration du périphérique (GPU / CPU)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Utilisation du périphérique : {device}")

    # 3. Instanciation du modèle ViT
    model = MyViT(
        chw=(1, 28, 28),
        n_patches=7,
        n_blocks=2,
        hidden_d=8,
        n_heads=2,
        out_d=10
    ).to(device)

    N_EPOCHS = 5
    LR = 0.005

    optimizer = Adam(model.parameters(), lr=LR)
    criterion = CrossEntropyLoss()

    # Initialisation des métriques torchmetrics
    test_acc_metric = Accuracy(task="multiclass", num_classes=10).to(device)
    test_f1_metric = F1Score(task="multiclass", num_classes=10, average='macro').to(device)

    # 4. Boucle d'entraînement
    print("\n--- Début de l'entraînement du ViT ---")
    start_train_time = time.time()

    for epoch in trange(N_EPOCHS, desc="Entraînement"):
        model.train()
        train_loss = 0.0

        for batch in tqdm(train_loader, desc=f"Époque {epoch + 1}", leave=False):
            x, y = batch
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            y_hat = model(x)
            loss = criterion(y_hat, y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * x.size(0)

        epoch_train_loss = train_loss / len(train_loader.dataset)
        print(f"Époque [{epoch + 1}/{N_EPOCHS}] - Train Loss: {epoch_train_loss:.4f}")

    training_time = time.time() - start_train_time
    print(f"Temps d'entraînement total : {training_time:.2f} secondes")

    # 5. Boucle d'évaluation (Test)
    print("\n--- Évaluation du ViT ---")
    model.eval()
    test_loss = 0.0
    test_acc_metric.reset()
    test_f1_metric.reset()

    start_eval_time = time.time()
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Test"):
            x, y = batch
            x, y = x.to(device), y.to(device)

            y_hat = model(x)
            loss = criterion(y_hat, y)
            test_loss += loss.item() * x.size(0)

            _, preds = torch.max(y_hat, dim=1)
            test_acc_metric.update(preds, y)
            test_f1_metric.update(preds, y)

    final_test_loss = test_loss / len(test_loader.dataset)
    final_test_acc = test_acc_metric.compute()
    final_test_f1 = test_f1_metric.compute()
    evaluation_time = time.time() - start_eval_time

    # Affichage final des métriques requises pour le comparatif du TP
    print("\n" + "="*40)
    print(" RÉSULTATS FINAUX - VISION TRANSFORMER (ViT) ")
    print("="*40)
    print(f"Loss (Test)     : {final_test_loss:.4f}")
    print(f"Accuracy (Test) : {final_test_acc.item():.4f}")
    print(f"F1-Score (Test) : {final_test_f1.item():.4f}")
    print(f"Temps d'éval    : {evaluation_time:.2f} secondes")
    print("="*40)

if __name__ == '__main__':
    main()

Utilisation du périphérique : cuda

--- Début de l'entraînement du ViT ---


Entraînement:   0%|          | 0/5 [00:00<?, ?it/s]

Époque 1:   0%|          | 0/469 [00:00<?, ?it/s]

Époque [1/5] - Train Loss: 1.6472


Époque 2:   0%|          | 0/469 [00:00<?, ?it/s]

Époque [2/5] - Train Loss: 0.7693


... (4 outputs left out)

Époque 5:   0%|          | 0/469 [00:00<?, ?it/s]

Époque [5/5] - Train Loss: 0.4430
Temps d'entraînement total : 1632.25 secondes

--- Évaluation du ViT ---


Test:   0%|          | 0/79 [00:00<?, ?it/s]


 RÉSULTATS FINAUX - VISION TRANSFORMER (ViT) 
Loss (Test)     : 0.3870
Accuracy (Test) : 0.8792
F1-Score (Test) : 0.8773
Temps d'éval    : 28.90 secondes
